# EDA: Exploratory Data Analysis

Khảo sát dataset trước khi train. Outputs sẽ dùng trong báo cáo:
- Số ảnh train/val
- Class distribution (cân hay không cân?)
- Bbox size distribution (ảnh hưởng tới imgsz)
- Sample visualization

In [ ]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent))

from collections import Counter
import cv2
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

from src.utils.io import load_yaml

sns.set_theme(style='whitegrid')

In [ ]:
# Đổi thành dataset của bạn
DATA_YAML = '../configs/data/players.yaml'
cfg = load_yaml(DATA_YAML)
DATASET_ROOT = Path(cfg['path']).resolve() if Path(cfg['path']).is_absolute() else (Path(DATA_YAML).parent / cfg['path']).resolve()
print(f'Dataset root: {DATASET_ROOT}')
print(f'Classes: {cfg["names"]}')

## 1. Count images per split

In [ ]:
splits = ['train', 'val']
counts = {}
for s in splits:
    img_dir = DATASET_ROOT / 'images' / s
    if img_dir.exists():
        counts[s] = len(list(img_dir.glob('*.jpg'))) + len(list(img_dir.glob('*.png')))
pd.Series(counts).plot(kind='bar', title='Images per split')
plt.ylabel('Count')
plt.show()
print(counts)

## 2. Class distribution

In [ ]:
class_counter = Counter()
bbox_sizes = []  # (w, h) normalized

for txt in (DATASET_ROOT / 'labels' / 'train').glob('*.txt'):
    for line in txt.read_text().strip().splitlines():
        if not line:
            continue
        parts = line.split()
        cls_id = int(parts[0])
        w, h = float(parts[3]), float(parts[4])
        class_counter[cls_id] += 1
        bbox_sizes.append((w, h))

names = cfg['names']
labels = [names[k] for k in sorted(class_counter.keys())]
values = [class_counter[k] for k in sorted(class_counter.keys())]

fig, ax = plt.subplots(figsize=(8, 4))
ax.bar(labels, values)
ax.set_title('Class distribution (train set)')
ax.set_ylabel('Bbox count')
for i, v in enumerate(values):
    ax.text(i, v, str(v), ha='center', va='bottom')
plt.tight_layout()
plt.savefig('../reports/figures/class_distribution.png', dpi=150)
plt.show()

## 3. Bbox size distribution

Nếu objects nhỏ (mean width < 0.05) → cần imgsz lớn hơn (1280 thay vì 640).

In [ ]:
bbox_df = pd.DataFrame(bbox_sizes, columns=['w', 'h'])
bbox_df['area'] = bbox_df['w'] * bbox_df['h']

fig, axes = plt.subplots(1, 3, figsize=(15, 4))
sns.histplot(bbox_df['w'], bins=50, ax=axes[0])
axes[0].set_title('Bbox width (normalized)')
sns.histplot(bbox_df['h'], bins=50, ax=axes[1])
axes[1].set_title('Bbox height (normalized)')
sns.histplot(np.sqrt(bbox_df['area']), bins=50, ax=axes[2])
axes[2].set_title('Bbox sqrt(area)')
plt.tight_layout()
plt.savefig('../reports/figures/bbox_size_dist.png', dpi=150)
plt.show()

print(bbox_df.describe())

## 4. Sample visualization với bbox

In [ ]:
def draw_yolo_labels(img, label_path, names):
    h, w = img.shape[:2]
    out = img.copy()
    colors = [(255,0,0), (0,255,0), (0,0,255), (255,255,0)]
    for line in label_path.read_text().strip().splitlines():
        cls_id, cx, cy, bw, bh = line.split()
        cls_id = int(cls_id)
        cx, cy, bw, bh = map(float, (cx, cy, bw, bh))
        x1 = int((cx - bw/2) * w)
        y1 = int((cy - bh/2) * h)
        x2 = int((cx + bw/2) * w)
        y2 = int((cy + bh/2) * h)
        c = colors[cls_id % len(colors)]
        cv2.rectangle(out, (x1, y1), (x2, y2), c, 2)
        cv2.putText(out, names.get(cls_id, str(cls_id)), (x1, y1-5), cv2.FONT_HERSHEY_SIMPLEX, 0.5, c, 1)
    return out

img_files = list((DATASET_ROOT / 'images' / 'train').glob('*.jpg'))[:6]
fig, axes = plt.subplots(2, 3, figsize=(15, 8))
for ax, p in zip(axes.flat, img_files):
    img = cv2.imread(str(p))
    label_path = DATASET_ROOT / 'labels' / 'train' / (p.stem + '.txt')
    if label_path.exists():
        img = draw_yolo_labels(img, label_path, cfg['names'])
    ax.imshow(cv2.cvtColor(img, cv2.COLOR_BGR2RGB))
    ax.set_title(p.name)
    ax.axis('off')
plt.tight_layout()
plt.savefig('../reports/figures/sample_labels.png', dpi=120)
plt.show()